In [1]:
#Load data
import torch
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, logging, RobertaModel

from dataset import CL_Dataset, CustomSubset
from evaluation import compute_cdcr_metrics
from model import SiameseNetwork


#Load model and tokenizer
model_path_entities = "models/mention_type_entity/model_roberta-large/siamese_model.pth"
model_path_events = "models/mention_type_entity/model_roberta-large/siamese_model.pth"

language_model = "roberta-large" # I should read it from the model's path
torch_device =  "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(torch_device)


# Initialize baselanguage model
base_language_model = RobertaModel.from_pretrained(pretrained_model_name_or_path= language_model)

# Initialize tokenizer
special_tokens = ['[E]', '[/E]']
tokenizer = RobertaTokenizer.from_pretrained(language_model)
tokenizer.add_tokens(special_tokens)

#Initialize siamese model
model_entities = SiameseNetwork(model=base_language_model,tokenizer=tokenizer,device=device).to(device)
model_events = SiameseNetwork(model=base_language_model,tokenizer=tokenizer,device=device).to(device)

model_entities.load_state_dict( torch.load(model_path_entities, map_location=torch.device(device)))
model_events.load_state_dict(torch.load(model_path_events, map_location=torch.device(device)))

model_entities.eval()
model_events.eval()
#Update weights given the chosen model


test_batch_size=128

c:\ProgramData\miniconda3\envs\mbib\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at roberta-large were not used when initializing RobertaModel: ['lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.bias', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaModel were not initialized from the mode

In [2]:
# Data
ecbplus_test_entities = CL_Dataset(name="ecbplus_test",data_paths=['../CDCR/data/ECBplus-prep/output_data/test'],context_method='hsu',
                    mention_type="entity",tokenizer=tokenizer, special_mention_tokens=special_tokens,mode="inference") 

ecbplus_test_entities = CustomSubset(ecbplus_test_entities,[0,1,2,3],mode="inference")

ecbplus_test_events = CL_Dataset(name="ecbplus_test",data_paths=['../CDCR/data/ECBplus-prep/output_data/test'],context_method='hsu',
                    mention_type="entity",tokenizer=tokenizer, special_mention_tokens=special_tokens,mode="inference") 

#Subset

newswcl50_entities = CL_Dataset(name="newswcl50_full",data_paths=['../CDCR/data/NewsWCL50-prep/output_data/train',
                                                                '../CDCR/data/NewsWCL50-prep/output_data/val',
                                                                '../CDCR/data/NewsWCL50-prep/output_data/test'],
                                                                context_method='hsu',mention_type="entity",tokenizer=tokenizer, 
                                                                special_mention_tokens=special_tokens,mode="inference") 

newswcl50_events = CL_Dataset(name="newswcl50_full",data_paths=['../CDCR/data/NewsWCL50-prep/output_data/train',
                                                                '../CDCR/data/NewsWCL50-prep/output_data/val',
                                                                '../CDCR/data/NewsWCL50-prep/output_data/test'],
                                                                context_method='hsu',mention_type="entity",tokenizer=tokenizer, 
                                                                special_mention_tokens=special_tokens,mode="inference") 

We need to remove 125 to truncate the input but the first sequence has a length 124. Please select another truncation strategy than only_first, for instance 'longest_first' or 'only_second'.
We need to remove 124 to truncate the input but the first sequence has a length 124. Please select another truncation strategy than only_first, for instance 'longest_first' or 'only_second'.
We need to remove 126 to truncate the input but the first sequence has a length 124. Please select another truncation strategy than only_first, for instance 'longest_first' or 'only_second'.


mention 8_L_4_57_AfSE ignored due to truncation of mention tokens
mention 8_L_4_7_6KXb ignored due to truncation of mention tokens
mention 8_L_4_32_Y8Ho ignored due to truncation of mention tokens


We need to remove 125 to truncate the input but the first sequence has a length 124. Please select another truncation strategy than only_first, for instance 'longest_first' or 'only_second'.
We need to remove 124 to truncate the input but the first sequence has a length 124. Please select another truncation strategy than only_first, for instance 'longest_first' or 'only_second'.
We need to remove 126 to truncate the input but the first sequence has a length 124. Please select another truncation strategy than only_first, for instance 'longest_first' or 'only_second'.


mention 8_L_4_57_AfSE ignored due to truncation of mention tokens
mention 8_L_4_7_6KXb ignored due to truncation of mention tokens
mention 8_L_4_32_Y8Ho ignored due to truncation of mention tokens


In [3]:
test_batch_size=16

In [4]:
# Prediction must be the same as what I did for validation

ecbplus_test_dataloader= DataLoader(ecbplus_test_entities, batch_size=test_batch_size, shuffle=False)

In [5]:
df_metrics, df_merged = compute_cdcr_metrics(cl_cdcr_dataset=ecbplus_test_entities,dataloader=ecbplus_test_dataloader,model=model_entities,metrics="all")

In [6]:
df_merged

,mention_id,subtopic_id,coref_chain,mention_type,gold_cluster,predicted_cluster,topic_id
0,36_1ecb_LOC17642835073942721_0_6_5nL6,36ecb,LOC17642835073942721,entity,0,0,36
1,36_1ecb_LOC17642835073942721_1_7_2zxG,36ecb,LOC17642835073942721,entity,0,0,36
2,36_2ecb_LOC17642835073942721_0_3_LMqr,36ecb,LOC17642835073942721,entity,0,0,36
3,36_8ecb_LOC17642835073942721_1_12_VKKB,36ecb,LOC17642835073942721,entity,0,0,36


In [16]:
#Test later on validation and compare metrics
df_metrics

,Recall,Precision,F1
Metric,,,
mentions,100.000000,100.000000,100.000000
muc,84.834123,81.628664,83.200531
bcub,83.027772,74.301435,78.422598
ceafe,76.587549,80.674097,78.577727
lea,70.892610,64.794858,67.706719
conll,NaN,NaN,80.066952
